# Chunk synthetic documents with traceable boundaries and idempotent IDs

**Audience:** Python developers preparing reproducible data workflows. **Time:** about 15–25 minutes. **Execution:** offline; no credentials, paid APIs, or downloaded datasets.

Build a small ingestion pipeline from authored documents to overlapping word chunks, preserve offsets, and verify re-ingestion does not duplicate records.

## Prerequisites and setup

Use the tested Python environment and exact dependency versions in the [collection setup](../../../README.md). Install from the collection root before opening this notebook, then select that environment's kernel. This notebook uses Python standard library and pandas. Restart the kernel and run all cells in order. No working-directory assumptions or private files are required.

## Data provenance

All examples below are newly authored synthetic fixtures. Names, identifiers, coordinates and values are invented for this tutorial and do not represent real people, accounts, organizations or operational records. Code is newly authored for this example; the references explain the underlying APIs.

Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Define documents and a transparent tokenization rule

This tutorial defines a token as a whitespace-delimited word. It is deliberately not a model tokenizer. The example teaches boundaries, provenance and stable identifiers before connecting any external vector database.

In [ ]:
from hashlib import sha256

import pandas as pd

documents = [
    {
        "document_id": "guide-1",
        "text": "Inspect the input carefully before processing. Keep a record of each transformation. Validate the final result before sharing.",
    },
    {
        "document_id": "guide-2",
        "text": "Use a small sample first. Check failures as well as successes. Repeat the workflow with the same configuration.",
    },
]
assert len({doc["document_id"] for doc in documents}) == len(documents)
pd.DataFrame(documents)

## 2. Chunk with half-open word offsets

`start_word` is inclusive and `end_word` is exclusive. The step is `size - overlap`, which must remain positive. IDs include the document ID, source digest, offsets and algorithm version so a changed document cannot silently reuse stale chunk IDs.

In [ ]:
def chunk_document(document, size=8, overlap=2):
    if type(size) is not int or type(overlap) is not int or size < 1 or not 0 <= overlap < size:
        raise ValueError("Require integer size > overlap >= 0")
    if not isinstance(document.get("document_id"), str) or not document["document_id"]:
        raise ValueError("Document ID required")
    text = document["text"]
    if not isinstance(text, str):
        raise ValueError("Document text must be a string")
    words = text.split()
    source_hash = sha256(text.encode()).hexdigest()
    chunks = []
    for start in range(0, len(words), size - overlap):
        end = min(start + size, len(words))
        identity = f"word-v1\0{document['document_id']}\0{source_hash}\0{start}\0{end}"
        chunks.append(
            {
                "chunk_id": sha256(identity.encode()).hexdigest(),
                "document_id": document["document_id"],
                "source_sha256": source_hash,
                "start_word": start,
                "end_word": end,
                "text": " ".join(words[start:end]),
            }
        )
        if end == len(words):
            break
    return chunks


chunks = [chunk for doc in documents for chunk in chunk_document(doc)]
assert len(chunks) == 6
pd.DataFrame(chunks)[["document_id", "start_word", "end_word", "text"]]

## 3. Verify coverage and overlap against the source

The checks compare each chunk to its source slice, then verify every source position appears at least once. This catches both missing tail words and off-by-one errors. Overlap duplicates some words intentionally.

In [ ]:
for doc in documents:
    words = doc["text"].split()
    group = [chunk for chunk in chunks if chunk["document_id"] == doc["document_id"]]
    covered = set()
    for chunk in group:
        start, end = chunk["start_word"], chunk["end_word"]
        assert chunk["text"] == " ".join(words[start:end])
        assert 0 < end - start <= 8
        covered.update(range(start, end))
    assert covered == set(range(len(words)))
    for previous, current in zip(group, group[1:]):
        assert previous["end_word"] - current["start_word"] == 2
print("All source words covered; offsets and overlap verified.")

## 4. Make ingestion idempotent

A dictionary demonstrates upsert semantics on real generated chunks. The second run adds zero new records. A changed document requires removing the old revision's chunks as part of a transaction; stable IDs alone do not implement garbage collection.

In [ ]:
index = {}


def upsert_chunks(items):
    inserted = 0
    for item in items:
        inserted += item["chunk_id"] not in index
        index[item["chunk_id"]] = dict(item)
    return inserted


assert upsert_chunks(chunks) == len(chunks)
assert upsert_chunks(chunks) == 0
assert len(index) == len(chunks)
assert chunk_document({"document_id": "empty", "text": ""}) == []
assert len(chunk_document({"document_id": "short", "text": "one two"})) == 1
for size, overlap in [(0, 0), (8, 8), (8, -1)]:
    try:
        chunk_document(documents[0], size, overlap)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid overlap accepted")
print(
    {"documents": len(documents), "chunks": len(index), "repeat_insertions": upsert_chunks(chunks)}
)

## Validation and interpretation

The assertions above are executable acceptance criteria. A failed assertion is a failed example: inspect the inputs and kernel versions before continuing. Inspect the displayed tables or figures as well; successful execution alone does not establish fitness for a different dataset.

## Limitations and extensions

No embeddings or vector service is invoked. Word overlap is a baseline, not a proven optimal retrieval strategy. Real deployments should compare sentence/section-aware chunking, use the embedding model tokenizer, enforce tenant separation and atomically retire old revisions. Evaluate retrieval quality on held-out questions before tuning chunk size.

## Cleanup

This notebook creates no persistent files, network sessions or cloud resources. Restart the kernel to discard in-memory objects. If you adapt it to real data, keep inputs outside a public repository and review every output before sharing.

## References and attribution

- [Python string splitting](https://docs.python.org/3/library/stdtypes.html#str.split)
- [Python hashlib](https://docs.python.org/3/library/hashlib.html)

The example is prepared for editorial review; it is not an IBM endorsement or a claim of production certification.